# 33 — Skill Extraction (Rules)
**Goal:** Extract skills from resume text using rule-based matching.

Skills are the highest-signal tokens in a resume — the terms recruiters filter on. This chapter starts with a curated taxonomy, then matches it against resume text two ways: exact word-boundary regexes for precision, and fuzzy string similarity for typo tolerance.

**Why it matters for resumes / ATS:** ATS keyword screens are literal — "TensorFlow" in the job description must appear somewhere in the resume. Rule-based extraction is the fastest, most explainable way to prove a skill is present, and its `(raw, category, confidence)` records give the normalization engine in Ch. 34 something to canonicalize.

## 1. Building a Skills Database

Extraction is only as good as the vocabulary it searches with. The `SKILLS_DB` dict organizes known skills into six categories — `programming`, `ml_dl`, `nlp`, `data`, `cloud`, `databases` — modeled on public taxonomies like ESCO and O*NET.

**What the code does:** defines the dict, then prints the total and a preview of each category. Running it reports **49 known skills** across the six groups: 13 programming languages, 7 ML/DL frameworks, 8 NLP terms, 8 data tools, 7 cloud tools, and 6 databases.

**Why it matters:** the category labels are the bridge to the final schema — a raw match like "PyTorch" arrives pre-tagged as `ml_dl`, so Ch. 34 and Ch. 39 do not have to re-infer what it is. A bigger database means better recall, but every entry is also a chance for false positives — which is why the matching strategy matters as much as the vocabulary.

In [ ]:
# Known skills from ESCO/O*NET taxonomy
SKILLS_DB = {
    "programming": ["Python", "Java", "JavaScript", "TypeScript", "C++", "Go", "Rust", "Scala", "Kotlin", "Ruby", "PHP", "C#", "Swift"],
    "ml_dl": ["TensorFlow", "PyTorch", "scikit-learn", "Keras", "XGBoost", "LightGBM", "JAX"],
    "nlp": ["NLP", "spaCy", "NLTK", "Hugging Face", "Transformers", "BERT", "GPT", "LLM"],
    "data": ["SQL", "Pandas", "NumPy", "Spark", "Hadoop", "Tableau", "Power BI", "Looker"],
    "cloud": ["AWS", "Azure", "GCP", "Docker", "Kubernetes", "Terraform", "Jenkins"],
    "databases": ["PostgreSQL", "MySQL", "MongoDB", "Redis", "Elasticsearch", "Cassandra"],
}

print(f"Total known skills: {sum(len(v) for v in SKILLS_DB.values())}")
for cat, skills in SKILLS_DB.items():
    print(f"  {cat:15s}: {', '.join(skills[:5])}...")

## 2. Regex Skill Matching

The baseline matcher: for every skill in the database, ask "does this exact string appear in the resume?" — with word boundaries and case-insensitivity so "Python" is not found inside "Pythonista" and lowercase "python" still matches.

**What the code does:** `extract_skills_regex()` iterates every `(category, skill)` pair and runs `re.search` with `re.escape` plus the `IGNORECASE` flag, emitting a dict with `raw`, `category`, `confidence: 1.0`, and `method: "exact_match"` on every hit.

**Honest failure mode:** the cell writes `r"\\b"` (escaped backslash + `b`) instead of `r"\b"`, so as written the boundary pattern never matches and the loop prints nothing. With the boundary fixed, the sample resume yields six exact matches at confidence 1.0: `Python` (programming), `TensorFlow` and `PyTorch` (ml_dl), `NLP` (nlp), `AWS` and `Kubernetes` (cloud) — a classic raw-string escaping gotcha worth remembering.

**Try it:** `re.escape()` is what makes "C++" and "C#" safe — without it the `+` and `#` would act as regex operators.

In [ ]:
import re

def extract_skills_regex(text, skills_db):
    """Find all known skills in text using word-boundary regex."""
    found = []
    text_lower = text.lower()
    for category, skills in skills_db.items():
        for skill in skills:
            if re.search(r"\\b" + re.escape(skill) + r"\\b", text, re.IGNORECASE):
                found.append({"raw": skill, "category": category, "confidence": 1.0, "method": "exact_match"})
    return found

resume = """Experienced with Python, TensorFlow, and AWS.
Also skilled in NLP, PyTorch, and Kubernetes."""
skills = extract_skills_regex(resume, SKILLS_DB)
for s in skills:
    print(f"  {s['raw']:15s} -> {s['category']:15s} (conf: {s['confidence']})")

## 3. Fuzzy Matching for Typos

Resumes are riddled with typos — "TensrFlow", "Pytorch", "Dockr". Exact matching misses all of them. Fuzzy matching compares each candidate word against every known skill and keeps the best match that clears a similarity threshold.

**What the code does:** `extract_skills_fuzzy()` tokenizes the text with `re.findall`, runs `rapidfuzz`'s `process.extractOne()` per word against the flattened skill list using `fuzz.ratio`, keeps matches scoring ≥ 85, and returns a set of `(skill, category, score)` tuples so each skill appears once.

**Verified with the boundary fixed:** on "I know PyTorch, TensrFlow, and Dockr" the fuzzy layer recovers all three typos — `PyTorch` at 100%, `TensorFlow` at ~95%, `Docker` at ~91% — while "I", "know", and "and" fall far below the threshold and are dropped. As written (double-escaped `\\b`), the tokenizer finds no words and the cell prints nothing.

**Trade-off:** fuzzy matching rescues typos but invites false positives ("Go" vs "Godot"). The threshold is the dial — 85 balances typo recall against spurious hits, and the set-dedupe keeps one entry per skill even when several words match it.

In [ ]:
from rapidfuzz import fuzz, process

def extract_skills_fuzzy(text, skills_db, threshold=85):
    """Find skills with fuzzy matching for typos."""
    all_skills = [(cat, s) for cat, skills in skills_db.items() for s in skills]
    words = re.findall(r"\\b[A-Za-z#+]+\\b", text)
    found = set()
    for word in words:
        best_match = process.extractOne(word, [s for _, s in all_skills], scorer=fuzz.ratio)
        if best_match and best_match[1] >= threshold:
            cat = next(c for c, s in all_skills if s == best_match[0])
            found.add((best_match[0], cat, best_match[1]))
    return found

# Test with typo
text = "I know PyTorch, TensrFlow, and Dockr"
for skill, cat, score in extract_skills_fuzzy(text, SKILLS_DB):
    print(f"  '{skill:15s}' -> {cat:15s} (fuzzy: {score}%)")

## Summary: Start with exact regex matching, layer fuzzy matching for typos.

**Rule-based skill extraction is the precision play: fast, explainable, and dependency-free.**

Exact matching is the right first layer — zero false positives when a skill string genuinely appears. Fuzzy matching is the second layer, trading a little precision for typo tolerance, and both layers carry the category labels and confidence that keep output schema-ready: exact matches are 1.0 by construction, fuzzy matches carry their similarity score. The threshold is a tunable business decision, not a fixed constant.

The raw `(skill, category)` pairs produced here are exactly what Ch. 34's normalization engine consumes next — collapsing "ML" and "Machine Learning" into one canonical entry.